# 1、加载数据，把数据处理成TRL SFTTrainer所需要的格式

In [ ]:
from datasets import load_dataset, DatasetDict
## 1.1 数据加载
data:DatasetDict = load_dataset("json",data_files={"train":"data/keywords_data_train.jsonl","test":"data/keywords_data_test.jsonl"})
data["train"] = data["train"].select(range(10000))

In [5]:
data

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 500
    })
})

In [6]:
## 1.2 数据处理
from typing import Dict,List
def convert_type(examples:Dict[str, List]):
    """
    讲数据，转换成 SFTTrainer所需要的Language Modeling类型，对话格式
    """
    conversation_list:List[List[Dict]] = examples["conversation"]

    all_data_messages_list = []

    for data in conversation_list:
        human_message = data[0]["human"]
        assistant_message = data[0]["assistant"]

        message_list = [
            {"role":"user","content":human_message},
            {"role":"assistant","content":assistant_message}
        ]

        all_data_messages_list.append(message_list)

    return {"messages":all_data_messages_list}


# batched=True，传递给convert_type的是一批数据，
mapped_data = data.map(convert_type,batched=True,remove_columns=['conversation_id', 'category', 'conversation', 'dataset'])

Map: 100%|██████████| 500/500 [00:00<00:00, 28487.72 examples/s]


In [8]:
mapped_data["train"][0]

{'messages': [{'content': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词',
   'role': 'user'},
  {'content': '高氟铍矿;配料;熔炼;回收率;脱氟率', 'role': 'assistant'}]}

## 1.2 获取数据的长度分布情况

In [9]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B/")
def get_length(examples:Dict[str,List]):
    """
    获取分词之后的每个样本的token_ids 序列的长度
    """

    # message_list 是多个样本组成的list
    # 第一个list是样本维度，第二个list是，实际单条样本的message列表
    message_list:List[List] = examples["messages"]
    length_list = []
    for data in message_list:
        # 此处data是单条样本的message列表
        input_ids_list = tokenizer.apply_chat_template(data,tokenize=True)["input_ids"]
        length_list.append(len(input_ids_list))

    return {"length":length_list}

length_data = mapped_data.map(get_length,batched=True, remove_columns=["messages"])
length_data

Map: 100%|██████████| 500/500 [00:00<00:00, 2821.96 examples/s]


DatasetDict({
    train: Dataset({
        features: ['length'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['length'],
        num_rows: 500
    })
})

In [13]:
length_data_df = length_data["train"].to_pandas()

In [19]:
length_data_df["length"].quantile(0.999)

np.float64(642.0320000000065)

# 2、构造SFTConfig对象

In [25]:
from trl.trainer.sft_config import SFTConfig
import os
os.environ["TENSORBOARD_LOGGING_DIR"]="./logs/05_sft_demo"

config = SFTConfig(
    per_device_train_batch_size=4,
    per_device_eval_batch_size= 4,
    gradient_accumulation_steps=8,
    max_steps=300,
    logging_strategy="steps",
    logging_steps=10,
    report_to="tensorboard",
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    eval_strategy="steps",
    eval_steps=50,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    output_dir="./finetuned/05_sft_demo",
    bf16=True,
    gradient_checkpointing=False,
    activation_offloading=False,
    max_length= 650,
    assistant_only_loss=True,
    chat_template_path="./new_chat_template.jinja"
)

# 3、构造SFTTrainer对象

In [26]:
from trl.trainer.sft_trainer import SFTTrainer
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B/")
trainer = SFTTrainer(
    model=model,
    args=config,
    processing_class=tokenizer,
    train_dataset=mapped_data["train"],
    eval_dataset=mapped_data["test"],
)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1354.71it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Truncating eval dataset: 100%|██████████| 500/500 [00:00<00:00, 27435.27 examples/s]


## 3.1 查看训练数据

In [28]:
data_loader = trainer.get_train_dataloader()

for batch in data_loader:
    print(tokenizer.decode(batch["input_ids"][0]))
    break

<|im_start|>user
全球化背景下,嵌入全球价值链的地方产业集群需要通过整合区域内外资源来实现自身的升级.基于全球价值链的研究视角,以福州电子信息产业集群为例剖析地方电子信息产业集群升级过程中遇到的困难和问题.结果表明:对外资的过分依赖,区域网络体系和支撑环境的薄弱是制约产业地方电子信息产业集群升级的重要因素.由此得出,在全球开放系统内培育一个不断发展和演变的总体支撑体系是地方电子信息产业集群升级的关键.
关键词抽取<|im_end|>
<|im_start|>assistant
<think>

</think>

全球价值链;电子信息产业;产业集群升级;福州<|im_end|>

<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endofte

# 4、训练以及模型的保存

In [ ]:
trainer.train()
# 保存模型参数，和Tokenizer相关的配置，从而使得，后面，可以通过加载这个路径，得到model和tokenizer
trainer.save_model("./finetuned/05_sft_demo")